# Solution 04 — consumer groups & manual offsets

Reference for the manual-commit pattern and the two-consumer experiment. The interesting bit isn't the code; it's the *behaviour* you should observe.

In [ ]:
from confluent_kafka import Consumer

BROKER = 'redpanda:29092'

consumer = Consumer({
    'bootstrap.servers':  BROKER,
    'group.id':           'manual-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,
})
consumer.subscribe(['rider-locations'])

messages_read, empty_polls = 0, 0
while messages_read < 5 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    print(f'#{messages_read} offset={msg.offset()} -- processing...', end='')
    consumer.commit(message=msg)
    print(' committed.')
consumer.close()

## Task A — observing re-delivery without commit

We deliberately *don't* commit. Run this twice in a row with the same `group.id` and you'll see the same offsets pop up both times.

In [ ]:
consumer = Consumer({
    'bootstrap.servers':  BROKER,
    'group.id':           'no-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,
})
consumer.subscribe(['rider-locations'])

messages_read, empty_polls = 0, 0
while messages_read < 3 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    print(f'#{messages_read} P{msg.partition()} offset={msg.offset()} -- (NOT committing)')
consumer.close()

## Task B — two consumers in one group

Run this cell in two tabs and produce events from a third. Each tab only prints the partitions it owns. When you stop one tab, the group rebalances and the other tab takes over all partitions after a few seconds.

In [ ]:
from confluent_kafka import Consumer
from datetime import datetime
import time

BROKER = 'redpanda:29092'
LISTEN_SECONDS = 300   # stops by itself after 5 minutes; stop earlier with ■

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'group-experiment',
    'auto.offset.reset': 'latest',         # only NEW events from now
})
consumer.subscribe(['rider-locations', 'orders'])
print('Listening — start the producer in another tab. Stop with the ■ button.')

deadline = time.time() + LISTEN_SECONDS
try:
    while time.time() < deadline:
        msg = consumer.poll(0.5)
        if msg is None or msg.error(): continue
        ts = datetime.now().strftime('%H:%M:%S')
        key = msg.key().decode() if msg.key() else 'None'
        print(f'[{ts}] {msg.topic()} P{msg.partition()} '
              f'offset={msg.offset()} key={key}')
    print('Time is up.')
except KeyboardInterrupt:
    print('\nStopped.')
finally:
    consumer.close()